In [11]:
import math
import numpy as np
import matplotlib.pyplot as plt
import random

In [12]:
class Value:

    def __init__(self, data, children=(), op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(children)
        self.op = op
        self.label = label

    def __repr__(self):
        return f'Value({self.data})'

    def __add__(self, other):
        other = other if type(other) == Value else Value(other)
        out = Value(self.data + other.data, (self, other,), '+')

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    def __radd__(self, other):
        return self + other

    def __mul__(self, other):
        other = other if type(other) == Value else Value(other)
        out = Value(self.data * other.data, (self, other,), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    def __rmul__(self, other):
        return self * other

    def __neg__(self, other):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):
        return other + (-self)

    def __pow__(self, other):
        assert isinstance(other, (int, float)), 'only supports int/float'
        out = Value(self.data**other, (self,), f'**{other}')

        def _backward():
            self.grad += out.grad * (other * self.data**(other-1))

        out._backward = _backward
        return out

    def __truediv__(self, other):
        return self * other**-1

    def exp(self):
        out = Value(math.exp(self.data), (self,), 'exp')

        def _backward():
            self.grad += out.grad * out.data

        out._backward = _backward
        return out

    def tanh(self):
        out = Value(math.tanh(self.data), (self,), 'tanh')

        def _backward():
            self.grad += out.grad * (1 - out.data**2)

        out._backward = _backward
        return out

    def backward(self):

        tops = []
        used = set()
        
        def dfs(v):
            if v not in used:
                used.add(v)
                for to in v._prev:
                    dfs(to)
                tops.append(v)

        self.grad = 1.0
        dfs(self)
        for v in reversed(tops):
            v._backward()


In [13]:
# inputs x1,x2
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')
# weights w1,w2
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')
# bias of the neuron
b = Value(6.8813735870195432, label='b')
# x1*w1 + x2*w2 + b
x1w1 = x1*w1; x1w1.label = 'x1*w1'
x2w2 = x2*w2; x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1*w1 + x2*w2'
n = x1w1x2w2 + b; n.label = 'n'
o = n.tanh(); o.label = 'o'

In [14]:
o

Value(0.7071067811865476)

In [15]:
class Neuron:

    def __init__(self, ins):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(ins)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        return (sum(wi*xi for wi, xi in zip(self.w, x)) + self.b).tanh()

    def parameters(self):
        return self.w + [self.b]

class Layer:

    def __init__(self, ins, outs):
        self.neurons = [Neuron(ins) for _ in range(outs)]

    def __call__(self, x):
        res = [neuron(x) for neuron in self.neurons]
        return res[0] if len(res) == 1 else res

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]

class MLP:

    def __init__(self, ins, szs):
        sz = [ins] + szs
        self.layers = [Layer(i, j) for i, j in zip(sz, sz[1:])]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [16]:
Perc = MLP(3, [4, 4, 1])

In [17]:
xs = [
  [2.0, 3.0, -1.0],
  [3.0, -1.0, 0.5],
  [0.5, 1.0, 1.0],
  [1.0, 1.0, -1.0],
]
ys = [1.0, -0.5, -1.0, 0.5]

In [18]:
for _ in range(100):
    y_pred = [Perc(x) for x in xs]

    loss = sum((y_p - y_t)**2 for y_p, y_t in zip(y_pred, ys))

    print(_, loss)
    
    for p in Perc.parameters():
        p.grad = 0.0

    loss.backward()

    for p in Perc.parameters():
        p.data -= 0.1 * p.grad

0 Value(8.30145969055409)
1 Value(3.926909585176708)
2 Value(4.99700257336195)
3 Value(2.570276708037625)
4 Value(2.507975053932011)
5 Value(2.498542691807379)
6 Value(2.490542822135167)
7 Value(2.483346089331777)
8 Value(2.4765723281214482)
9 Value(2.4699580450714382)
10 Value(2.463305245495323)
11 Value(2.456455743879567)
12 Value(2.4492772193665924)
13 Value(2.4416556036807204)
14 Value(2.433490916381684)
15 Value(2.424694382702118)
16 Value(2.415184543013015)
17 Value(2.404879521787474)
18 Value(2.393682072184109)
19 Value(2.3814538235347786)
20 Value(2.3679752485297056)
21 Value(2.35288698617105)
22 Value(2.3356034435117556)
23 Value(2.3151765973562295)
24 Value(2.2900593519943357)
25 Value(2.2576568644414228)
26 Value(2.213415939407822)
27 Value(2.148875064039469)
28 Value(2.0473907222574255)
29 Value(1.8757783351542625)
30 Value(1.5806924957981061)
31 Value(1.160916356814146)
32 Value(0.7375336422150197)
33 Value(0.4387259113169987)
34 Value(0.28149224843730003)
35 Value(0.20702

In [19]:
y_pred

[Value(0.8169653442405325),
 Value(-0.5009637334785342),
 Value(-0.9644324118082562),
 Value(0.5807595656068398)]